# Aula 06 — Memória: o agente que lembra entre mensagens

**Objetivo:** entender por que todo agente até aqui sofre de amnésia — e como
`checkpointer + thread_id` transformam `invoke`s isolados em conversa.

**Pré-requisitos:** Aula 05 (o loop ReAct) e Aula 02 (estado nasce do `invoke`).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` usa Groq real; sem chave, Fake offline.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: HTTP sem cookie vs comanda da lanchonete

Todo sistema web clássico que você já fez resolveu este problema: cada request
HTTP chega 'zerado', e quem precisa de continuidade usa **sessão** (cookie +
dado guardado no servidor).

| Web clássica | No LangGraph | O que é de verdade |
|---|---|---|
| request isolado | **`invoke` sem checkpointer** | estado nasce do input e morre no `END` |
| sessão (cookie) | **`thread_id`** | chave da conversa: 'sessão da Ana', 'mesa 12' |
| banco de sessão | **checkpointer** (`MemorySaver`) | guarda o estado **após cada passo** do grafo |
| reidratar sessão | passar `config` no `invoke` | o turno novo começa do estado salvo, não do zero |

Na lanchonete: sem comanda, cada ida ao balcão recomeça o pedido ('o que era
mesmo o `isso`?'). Com comanda (thread) + anotações a cada item (checkpoints),
o atendente continua de onde parou.

In [ ]:
# CÉLULA 2 — montando o agente (igual ao da Aula 05)
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

SISTEMA = 'Você é um assistente que faz contas. Responda em português, de forma curta.'
MODELO = 'openai/gpt-oss-20b'

def add(a: int, b: int) -> int:
    """Soma a e b."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiplica a por b."""
    return a * b

TOOLS = [add, multiply]

def get_llm():
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> Fake offline.')
        else:
            return ChatGroq(model=MODELO, temperature=0,
                            reasoning_effort='low').bind_tools(TOOLS)
    from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
    print('AVISO: modo Fake offline (sequência de 2 turnos gravada).')
    return FakeMessagesListChatModel(responses=[
        AIMessage(content='', tool_calls=[
            {'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'c1', 'type': 'tool_call'}]),
        AIMessage(content='7', tool_calls=[]),
        AIMessage(content='', tool_calls=[
            {'name': 'add', 'args': {'a': 7, 'b': 7}, 'id': 'c2', 'type': 'tool_call'}]),
        AIMessage(content='14', tool_calls=[]),
    ])

# O LLM é criado UMA vez aqui fora: se fosse criado dentro de
# assistant(), cada volta ganharia um Fake 'zerado' (sempre a 1ª fala)
# e o loop nunca terminaria. ChatGroq real não sofre disso (é stateless),
# mas a regra vale para os dois: construa fora, reuse dentro.
_llm = get_llm()

def assistant(state: MessagesState) -> dict:
    return {'messages': [_llm.invoke(
        [SystemMessage(content=SISTEMA)] + state['messages'])]}

def montar(checkpointer=None):
    b = StateGraph(MessagesState)
    b.add_node('assistant', assistant)
    b.add_node('tools', ToolNode(TOOLS))
    b.add_edge(START, 'assistant')
    b.add_conditional_edges('assistant', tools_condition)
    b.add_edge('tools', 'assistant')
    return b.compile(checkpointer=checkpointer)

print('agente montado (mesmo ReAct da Aula 05).')

## 2. Conceito: estado transitório vs estado com sessão

- **Sem checkpointer** (Aulas 02–05): o estado nasce do dicionário que você
  passa no `invoke` e morre no `END`. Dois `invoke`s são dois universos — o
  segundo nunca viu o primeiro. É o 'HTTP sem cookie'.
- **Com checkpointer**: `compile(checkpointer=MemorySaver())` cria o 'banco
  de sessão'. A cada passo do grafo (cada nó executado), o estado é gravado.
- **`config={'configurable': {'thread_id': 'sessao-1'}}`**: o 'cookie'. Diz
  *qual* conversa reidratar. Mesmo `thread_id` = continuar; diferente = nova.
- **Reidratação**: no `invoke` seguinte, o estado inicial não é só o seu
  `HumanMessage` novo — é o estado salvo **mais** sua mensagem anexada
  (via `add_messages`). Por isso 'isso' resolve para 7.

In [ ]:
# CÉLULA 3 — a prova do esquecimento (sem checkpointer)
sem_memoria = montar()   # sem checkpointer: cada invoke é um mundo novo

t1 = sem_memoria.invoke(
    {'messages': [HumanMessage(content='Quanto é 3 mais 4? Responda só o número.')]})
print('turno 1:', repr(t1['messages'][-1].content))

t2 = sem_memoria.invoke(
    {'messages': [HumanMessage(content='Multiplique isso por 2. Responda só o número.')]})
print('turno 2:', repr(t2['messages'][-1].content))
print('mensagens que o turno 2 viu:', len(t2['messages']))
assert len(t2['messages']) == 4  # só a própria execução; nada do turno 1
if os.getenv('GROQ_API_KEY'):
    print('(com Groq real, o turno 2 chuta ou pede o número: ele NÃO viu o 7.)')
else:
    print('(no Fake, a sequência gravada resolve; o que prova o esquecimento')
    print(' é a contagem: 4 mensagens, zero vestígio do turno 1.)')

## 3. Dissecando a sessão (linha a linha do que muda)

- `from langgraph.checkpoint.memory import MemorySaver` — o banco de sessão
  em RAM (rápido, some ao reiniciar o processo; produção usa SQLite/Postgres).
- `montar(checkpointer=MemorySaver())` — o MESMO builder, compilado com memória.
  Topologia idêntica; só a persistência muda.
- `cfg = {'configurable': {'thread_id': 'sessao-1'}}` — o identificador da
  conversa. Passe o **mesmo dict** como 2º argumento de **todo** `invoke`.
- `invoke({...}, cfg)` — o LangGraph carrega o checkpoint da thread, anexa sua
  mensagem nova e executa. O retorno já é o estado completo (turno 1 + turno 2).
- Sem `thread_id` num grafo com checkpointer: `ValueError` na cara (seção 5c).
  É de propósito — sessão anônima não existe.

In [ ]:
# CÉLULA 4 — com memória: 'isso' resolve para 7 (e vira 14)
com_memoria = montar(checkpointer=MemorySaver())
cfg = {'configurable': {'thread_id': 'sessao-1'}}

r1 = com_memoria.invoke(
    {'messages': [HumanMessage(content='Quanto é 3 mais 4? Responda só o número.')]}, cfg)
print('turno 1:', repr(r1['messages'][-1].content), '| estado:', len(r1['messages']), 'msgs')

r2 = com_memoria.invoke(
    {'messages': [HumanMessage(content='Multiplique isso por 2. Responda só o número.')]}, cfg)
print('turno 2:', repr(r2['messages'][-1].content), '| estado:', len(r2['messages']), 'msgs')
assert len(r2['messages']) == 8  # turno 1 (4) + turno 2 (4): reidratou
if os.getenv('GROQ_API_KEY'):
    assert '14' in r2['messages'][-1].content, r2['messages'][-1].content
    print('OK — com Groq real: 7 lembrado, 14 calculado.')
else:
    print('OK — no Fake: 8 mensagens provam a reidratação (o 14 é da sequência gravada).')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Thread diferente = conversa isolada (sem vazamento)

A 'mesa 12' não vê o pedido da 'mesa 7'. Sem isso, a Ana leria a conversa da
Bia — vazamento de dados entre usuários.

In [ ]:
# CÉLULA 5 — isolamento entre threads (Groq real e Fake)
cfg_ana = {'configurable': {'thread_id': 'ana'}}
cfg_bia = {'configurable': {'thread_id': 'bia'}}

com_memoria.invoke(
    {'messages': [HumanMessage(content='Quanto é 3 mais 4? Responda só o número.')]}, cfg_ana)
out_bia = com_memoria.invoke(
    {'messages': [HumanMessage(content='Multiplique isso por 2. Responda só o número.')]}, cfg_bia)
print('bia viu:', len(out_bia['messages']), 'msgs | final:', repr(out_bia['messages'][-1].content))
assert len(out_bia['messages']) == 4  # só a execução dela; nada da ana
assert all('3 mais 4' not in m.content for m in out_bia['messages'])
print('OK — isolamento: a thread bia nunca viu o turno da ana.')

### 4b. Inspecionando a sessão: `get_state` e `get_state_history`

- `get_state(cfg)` — o estado **atual** da thread (o que o próximo turno vai
  reidratar). `next` vazio `()` = grafo parado no `END`, pronto para continuar.
- `get_state_history(cfg)` — a lista de checkpoints (um por passo executado).
  É a 'fita' da conversa: dá para auditar, debugar e — na Aula de viagem no
  tempo — voltar atrás.

In [ ]:
# CÉLULA 6 — raio-X da sessão (100% offline na lógica)
atual = com_memoria.get_state(cfg)
print('estado atual: msgs =', len(atual.values['messages']), '| next =', atual.next)
historico = list(com_memoria.get_state_history(cfg))
print('checkpoints gravados:', len(historico))
assert len(historico) > 2  # um checkpoint por passo do grafo
print('OK — cada passo do grafo deixou um checkpoint (é assim que a volta no tempo funciona).')

### 4c. Sem `thread_id`, com checkpointer: `ValueError` na cara

Sessão anônima não existe: o banco precisa saber *onde* guardar. Esquecer o
`config` é o bug mais comum desta aula — o erro diz exatamente o que falta.

In [ ]:
# CÉLULA 7 — esquecer o config (100% offline)
try:
    com_memoria.invoke({'messages': [HumanMessage(content='oi')]})
    print('rodou?! (não deveria)')
except ValueError as erro:
    print('ValueError:', str(erro)[:110])
    print('OK — decore: grafo com checkpointer SEMPRE pede config com thread_id.')

## 5. Erros clássicos

1. **Passar `config` só no 1º turno.** Cada `invoke` precisa do `cfg` — sem ele,
   `ValueError` (CÉLULA 7). Vire hábito: `invoke(msg, cfg)` sempre juntos.
2. **Reusar `thread_id` entre usuários.** Mesma thread = mesma conversa. 'ana'
   e 'bia' no mesmo id = vazamento de dados (CÉLULA 5 mostra o isolamento).
3. **Achar que a memória é 'do LLM'.** O modelo não lembra nada entre chamadas;
   quem lembra é o checkpointer, reenviando o histórico a cada turno. Prova:
   troque de modelo no meio da conversa que o 'isso=7' continua valendo.
4. **Confiar no `MemorySaver` em produção.** Ele é RAM: reiniciou o processo,
   perdeu as sessões. Produção usa checkpointer persistente (SQLite/Postgres).
5. **Histórico infinito.** Cada turno cresce `messages`; em algum ponto estoura
   a janela do modelo. Podar/resumir é a Aula 08 (`trim_messages`, `RemoveMessage`).
6. **Afirmar demais no modo Fake.** O Fake desta aula tem a sequência gravada
   (7, depois 14) — ele não 'entendeu' nada. Fake prova mecânica, não inteligência.

## 6. Exercício guiado — a comanda da lanchonete em 2 turnos

Situação: turno 1, 'Quero 2 pastéis de R$ 8. Quanto dá?' (esperado: 16).
Turno 2, 'Adiciona 1 refri de R$ 6. E agora o total?' (esperado: 22).
O turno 2 só acerta se reidratar o turno 1 — é o teste de fogo da memória.

In [ ]:
# CÉLULA 8 — exercício guiado (2 turnos, mesma thread)
cfg_lanche = {'configurable': {'thread_id': 'mesa-12'}}
p1 = com_memoria.invoke(
    {'messages': [HumanMessage(content='Quero 2 pasteis de R$ 8. Quanto dá? Responda só o número.')]},
    cfg_lanche)
print('turno 1:', repr(p1['messages'][-1].content))
p2 = com_memoria.invoke(
    {'messages': [HumanMessage(content='Adiciona 1 refri de R$ 6. E agora o total? Responda só o número.')]},
    cfg_lanche)
print('turno 2:', repr(p2['messages'][-1].content), '| estado:', len(p2['messages']), 'msgs')
if os.getenv('GROQ_API_KEY'):
    assert '16' in p1['messages'][-1].content, p1['messages'][-1].content
    assert '22' in p2['messages'][-1].content, p2['messages'][-1].content
    print('OK — 16 no turno 1, 22 no turno 2: a comanda funcionou.')
else:
    print('(sem chave: o Fake resolve a sequência gravada;')
    print(' com a chave Groq, esta célula valida 16 e depois 22.)')

## 7. Desafio — 3 turnos com cancelamento

Na mesma thread, faça:

1. 'Quero 2 pastéis de R$ 8. Quanto dá?' → esperado `16`
2. 'Adiciona 1 refri de R$ 6. E agora?' → esperado `22`
3. 'Cancela o refri. Fecha a conta.' → esperado `16` de novo

O turno 3 referencia o turno 2 ('o refri') e invalida parte dele — sem memória
isso é impossível. Com a chave Groq, valide os 3 finais (`16`, `22`, `16`).
Sem chave: desenhe no papel quantas mensagens o estado teria ao final
(dica: cada turno ReAct desta aula gera 4 mensagens).

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Por que dois `invoke`s sem checkpointer são 'dois universos'?
- [ ] O que o checkpointer grava, quando grava, e o que o `thread_id` escolhe?
- [ ] O que acontece se esquecer o `config`? E se reusar a thread de outro usuário?
- [ ] Onde a memória mora: no LLM ou no checkpointer? Como provar?
- [ ] Por que o `MemorySaver` não serve para produção e o que usar no lugar?
- [ ] No seu exemplo do pastel/refri/conta, qual `thread_id` você usaria por cliente?

**Próxima aula (07 — Schemas):** `TypedDict` vs `dataclass` vs `Pydantic` —
contratos de estado, `Literal` como enum e validação que barra `mood='bravo'`.